# Credit Card Fraud Detection

**Dataset**: [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Problem type**: Binary Classification  
**Target**: `Class` — 0 = legitimate, 1 = fraud

**Dataset note**: Features V1–V28 are PCA-transformed — original transaction details are anonymized for privacy. Only `Time` and `Amount` are in their original form.

---

## Sections
1. First Look
2. Class Imbalance
3. Feature Distributions
4. Modeling
5. Evaluation & Threshold Tuning
6. Cross-Validation
7. Hyperparameter Tuning
8. Feature Importance
9. SHAP — Explainability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

df = pd.read_csv('creditcard.csv')

## 1. First Look

In [ ]:
df.head()

In [ ]:
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.info()

284,807 transactions, 30 features + 1 target. No missing values. All features are numeric — no encoding needed.

## 2. Class Imbalance

In [ ]:
class_counts = df['Class'].value_counts()
print(class_counts)
print(f"\nFraud rate: {class_counts[1] / len(df) * 100:.3f}%")

Only 492 fraud transactions out of 284,807 — fraud is 0.17% of all transactions. 

This is severe class imbalance. A model that predicts everything as legitimate would still get 99.8% accuracy — which is useless. We need to handle this before modeling using SMOTE.

## 3. Feature Distributions

### 3.1 Transaction Amount

Comparing amount distributions across classes. Using `stat='density'` and the same x-axis range so the comparison is fair — raw counts would be misleading given the class imbalance.

In [ ]:
limit = 2500

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df[(df['Class'] == 0) & (df['Amount'] <= limit)]['Amount'],
             bins=50, ax=axes[0], stat='density')
axes[0].set_title('Legitimate (Amount ≤ 2500)')

sns.histplot(df[(df['Class'] == 1) & (df['Amount'] <= limit)]['Amount'],
             bins=50, ax=axes[1], color='red', stat='density')
axes[1].set_title('Fraud (Amount ≤ 2500)')

plt.tight_layout()
plt.show()

Both distributions look nearly identical — `Amount` alone does not distinguish fraud from legitimate. Fraudsters deliberately blend in with normal transaction amounts to avoid detection.

### 3.2 V Features — KDE Comparison

KDE plots normalize each class independently (area under curve = 1), so class imbalance doesn't affect the comparison. We're comparing probabilities, not counts.

Features where the red (fraud) and blue (legitimate) curves are far apart are the most useful for the model.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(['V1','V2','V3','V4','V5','V6','V7','V8',
                          'V9','V10','V11','V12','V14','V16','V17','V18']):
    sns.kdeplot(df[df['Class'] == 0][col], ax=axes[i], label='Legit')
    sns.kdeplot(df[df['Class'] == 1][col], ax=axes[i], label='Fraud', color='red')
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.tight_layout()
plt.show()

**Strong features** (curves clearly separated): V4, V9, V10, V11, V12, V14, V16, V17  
**Weak features** (curves overlap): V7, V8

Legitimate transactions cluster tightly around 0 for most V features. Fraud transactions are shifted far from 0 — either strongly negative (V14, V10, V12, V17) or strongly positive (V4, V11).

## 4. Modeling

**Pipeline:**
1. Train-test split first
2. SMOTE applied only on training data — prevents data leakage
3. Random Forest trained on balanced data
4. Evaluated on original untouched test set

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_train_smote.value_counts().to_dict())

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_smote, y_train_smote)

y_pred = model.predict(X_test)

## 5. Evaluation & Threshold Tuning

Accuracy is meaningless for imbalanced data. We focus on:
- **Recall** — of all actual frauds, how many did we catch? (missing fraud = costly)
- **Precision** — of all flagged frauds, how many were real? (false alarms = customer friction)

Default threshold is 0.5. Lowering it makes the model more aggressive at flagging fraud — recall goes up, precision goes down.

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_pred_proba >= 0.3).astype(int)

print(classification_report(y_test, y_pred_adjusted))

cm = confusion_matrix(y_test, y_pred_adjusted)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (threshold=0.3)')
plt.show()

At threshold 0.3: **Recall = 0.89, Precision = 0.81** — the model catches 89% of all fraud while keeping false alarms low. This is the best balance point — lowering to 0.2 gave no recall gain but hurt precision significantly.

## 6. Cross-Validation

A single train-test split can be optimistic. Cross-validation trains and evaluates 5 times on different splits and averages the result — giving a more honest performance estimate.

SMOTE is placed inside the pipeline so it runs fresh on each training fold. Applying SMOTE before CV would cause data leakage — synthetic test points would contain information from the training fold.

In [ ]:
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

recall_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='recall')

print(f"Recall per fold: {recall_scores.round(3)}")
print(f"Mean Recall:     {recall_scores.mean():.3f}")
print(f"Std:             {recall_scores.std():.3f}")

Mean recall of ~0.815 across folds — more conservative than the single-split result but more reliable. The variation across folds is expected given only 394 fraud cases in training.

## 7. Hyperparameter Tuning

Grid search tries every combination of parameters and uses cross-validation to evaluate each one. Optimizing for recall since missing fraud is the costlier mistake.

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='recall', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"Best Recall: {grid_search.best_score_:.3f}")
print(f"Best Params: {grid_search.best_params_}")

In [ ]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print(classification_report(y_test, y_pred_best))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Best Model)')
plt.show()

Best params: `max_depth=10, min_samples_split=5, n_estimators=100`. Limiting tree depth reduced overfitting. 

The tuned model optimized purely for recall — same recall as threshold-adjusted model but with lower precision. For a balanced result, the original model at threshold=0.3 remains the better choice.

## 8. Feature Importance

Verifying the EDA hypothesis — KDE plots suggested V4, V10, V11, V12, V14, V17 were the strongest signals. Does the model agree?

In [ ]:
importances = best_model['model'].feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=feat_imp.values[:15], y=feat_imp.index[:15])
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.show()

Model confirms the EDA hypothesis. V14 is the single strongest feature (importance ~0.22), followed by V10, V12, V4, V17 — all flagged in the KDE analysis. V17 and V3 were slightly underestimated in EDA but the model picked them up.

The bottom features (V6, V19, V28) contribute almost nothing — the model ignores them.

## 9. SHAP — Explainability

Feature importance tells us which features matter globally. SHAP explains why the model made a specific prediction for one transaction — useful in production when a customer asks why their card was blocked.

SHAP uses the same trained model. For each transaction, it probes the model once per feature to measure how much each feature pushed the prediction toward fraud or legitimate.

In [ ]:
explainer = shap.TreeExplainer(best_model['model'])
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values[:, :, 1], X_test)

Each dot is one transaction. Color shows the feature value (red = high, blue = low). X-axis shows direction: right = pushed toward fraud, left = pushed toward legitimate.

- **V14, V10, V12, V17**: low values (blue) push toward fraud
- **V4, V11**: high values (red) push toward fraud
- Bottom features: dots clustered near zero — almost no influence on predictions

### Single Transaction Explanation

Waterfall plot for one specific fraud transaction — shows exactly how each feature contributed to the final prediction.

In [ ]:
fraud_index = X_test[y_test == 1].index[0]
single_transaction = X_test.loc[[fraud_index]]

shap_single = explainer.shap_values(single_transaction)

shap.waterfall_plot(
    shap.Explanation(
        values=shap_single[0][:, 1],
        base_values=explainer.expected_value[1],
        data=single_transaction.iloc[0],
        feature_names=X_test.columns.tolist()
    )
)

Reading the waterfall chart:
- **Base value (E[f(X)] = 0.5)** — starting point with no information
- Each bar shows one feature's contribution — positive bars push toward fraud, negative toward legitimate
- **f(x) = 0.999** — final fraud probability after all features stack

V14 alone pushed probability from 0.5 → 0.61. V17, V12, V4, V10 stacked on top until reaching 0.999. The 21 remaining features contributed almost nothing — confirming that a small set of features does most of the work.